Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install libraries

In [2]:
!pip install -q evaluate datasets transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00


Imports

In [3]:
import pandas as pd
import numpy as np
import ast
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

import evaluate

load dataset

In [5]:
dataset_path = "/content/drive/MyDrive/plant disease/plantvillage.parquet"

df = pd.read_parquet(dataset_path)

if "image" in df.columns:
    df = df.drop(columns=["image"])

df["captions"] = df["captions"].apply(
    lambda x: list(x) if isinstance(x, (list, np.ndarray)) else ast.literal_eval(x)
)

print("Rows before explode:", len(df))

Rows before explode: 20638


explode before split

In [6]:
# Explode
df = df.explode("captions", ignore_index=True)

df = df.rename(columns={"captions": "text"})
df["text"] = df["text"].astype(str)

df = df.dropna()

# Normalize
df["text"] = df["text"].str.strip().str.lower()

# 🔥 DO NOT REMOVE DATA — JUST SHUFFLE
df = df.sample(frac=1, random_state=42)

# 🔥 ADD PLANT CONTEXT (IMPORTANT)
def add_plant_context(row):
    plant = row["caption"].split()[0].lower()
    return f"{plant} leaf {row['text']}"

df["text"] = df.apply(add_plant_context, axis=1)

print("Total rows:", len(df))

Total rows: 82552


train-test split

In [7]:
train_df, test_df = train_test_split(
    df,
    test_size=0.3,   # 🔥 increase this
    random_state=42,
    stratify=df["caption"]
)

label encoding

In [8]:
encoder = LabelEncoder()

train_df["label"] = encoder.fit_transform(train_df["caption"])
test_df["label"] = encoder.transform(test_df["caption"])

num_classes = len(encoder.classes_)
print("Total classes:", num_classes)

Total classes: 15


convert to huggingface

In [9]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

tokenization

In [10]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/57786 [00:00<?, ? examples/s]

Map:   0%|          | 0/24766 [00:00<?, ? examples/s]

load model

In [12]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    dropout=0.3,               # ✅ correct
    attention_dropout=0.3      # ✅ correct
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


metrics

In [13]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

stable training

In [14]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,
    learning_rate=3e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.1,
    warmup_ratio=0.1,

    eval_strategy="epoch",        # ✅ FIXED
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    logging_steps=100,
    save_total_limit=2,

    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainer

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000263,0.000093,1.000000
2,0.000080,0.000027,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=7224, training_loss=0.1210317804641129, metrics={'train_runtime': 517.8875, 'train_samples_per_second': 223.16, 'train_steps_per_second': 13.949, 'total_flos': 957066969012480.0, 'train_loss': 0.1210317804641129, 'epoch': 2.0})

evaluate

In [17]:
output = trainer.predict(test_dataset)

preds = np.argmax(output.predictions, axis=1)

pred_labels = encoder.inverse_transform(preds)
true_labels = encoder.inverse_transform(test_df["label"].values)

for i in range(10):
    print("Text:", test_df["text"].iloc[i])
    print("True:", true_labels[i])
    print("Pred:", pred_labels[i])
    print("------")

print("Test Accuracy:", output.metrics["test_accuracy"])

Text: potato leaf a potato leaf infected with alternaria solani, displaying necrotic spots with dark concentric circles.
True: Potato Early blight
Pred: Potato Early blight
------
Text: pepper leaf a high-resolution image of a bell pepper leaf affected by bacterial spot, designed for plant disease classification.
True: Pepper bell Bacterial spot
Pred: Pepper bell Bacterial spot
------
Text: pepper leaf a healthy green bell pepper leaf with scattered dark spots, photographed in natural light.
True: Pepper bell Bacterial spot
Pred: Pepper bell Bacterial spot
------
Text: tomato leaf a tomato leaf infested with two-spotted spider mites, showing tiny webs and yellow stippling.
True: Tomato Spider mites Two spotted spider mite
Pred: Tomato Spider mites Two spotted spider mite
------
Text: tomato leaf a tomato leaf showing small, circular brown spots with yellow halos, typical of septoria leaf spot.
True: Tomato Septoria leaf spot
Pred: Tomato Septoria leaf spot
------
Text: tomato leaf a to

save and download

In [18]:
from google.colab import files

SAVE_DIR = "/content/drive/MyDrive/plant disease"
ZIP_NAME = "plant_disease_model_final.zip"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f"{SAVE_DIR}/classes.json", "w") as f:
    json.dump(encoder.classes_.tolist(), f)

print("Model saved at:", SAVE_DIR)

!zip -r {ZIP_NAME} "{SAVE_DIR}"

files.download(ZIP_NAME)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: /content/drive/MyDrive/plant disease
  adding: content/drive/MyDrive/plant disease/ (stored 0%)
  adding: content/drive/MyDrive/plant disease/plantvillage.parquet (deflated 4%)
  adding: content/drive/MyDrive/plant disease/config.json (deflated 61%)
  adding: content/drive/MyDrive/plant disease/model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/plant disease/tokenizer_config.json (deflated 42%)
  adding: content/drive/MyDrive/plant disease/training_args.bin (deflated 53%)
  adding: content/drive/MyDrive/plant disease/tokenizer.json (deflated 71%)
  adding: content/drive/MyDrive/plant disease/classes.json (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import torch
import torch.nn.functional as F
import json

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/plant disease"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

with open(f"{MODEL_PATH}/classes.json") as f:
    classes = json.load(f)

model.eval()

# ------------------------
# FIXED PREDICT FUNCTION
# ------------------------
def predict(text):

    words = text.lower().split()

    # 🔥 FIX DUPLICATION ISSUE
    plant = words[0]

    if "leaf" not in words:
        text = f"{plant} leaf {' '.join(words[1:])}"
    else:
        text = ' '.join(words)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # temperature scaling
    probs = F.softmax(outputs.logits / 2.0, dim=1)

    top_k = 3
    values, indices = torch.topk(probs, top_k)

    print(f"\nInput: {text}\n")

    for i in range(top_k):
        label = classes[indices[0][i].item()]
        score = values[0][i].item()
        print(f"{i+1}. {label} → {score:.4f}")

    if values[0][0].item() < 0.5:
        print("⚠️ Low confidence")


# ------------------------
# TEST SAMPLES
# ------------------------
samples = [
    "tomato yellow spots",
    "potato brown patches",
    "leaf white powder",
    "leaf curling",
    "insects under leaf"
]

for s in samples:
    predict(s)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Input: tomato leaf yellow spots

1. Tomato mosaic virus → 0.6911
2. Tomato Target Spot → 0.1149
3. Tomato Leaf Mold → 0.0405

Input: potato leaf brown patches

1. Potato Early blight → 0.6572
2. Potato healthy → 0.1251
3. Potato Late blight → 0.0975

Input: leaf white powder

1. Pepper bell healthy → 0.6935
2. Potato healthy → 0.0643
3. Tomato Leaf Mold → 0.0437

Input: leaf curling

1. Tomato YellowLeaf Curl Virus → 0.7140
2. Potato healthy → 0.0782
3. Tomato mosaic virus → 0.0450

Input: insects under leaf

1. Tomato Spider mites Two spotted spider mite → 0.7851
2. Potato Early blight → 0.0496
3. Tomato Early blight → 0.0341
